In [1]:
!pip install sentence-transformers torch

## Evaluation of embedding mode

MTEB (Massive Text Embedding Benchmark)

STSBenchmark = Semantic Textual Similarity Benchmark

| Metric                    | Tumhara Score | Typical Range | Quality Level    | Iska Meaning                                      | Practical Impact                                   |
| ------------------------- | ------------- | ------------- | ---------------- | ------------------------------------------------- | -------------------------------------------------- |
| **Spearman (Main Score)** | **0.8203**    | 0.75 – 0.90   | 🟢 Good Baseline | Model ranking human similarity ke kaafi close hai | Semantic search me mostly relevant results milenge |
| Pearson                   | 0.8274        | 0.75 – 0.90   | 🟢 Good          | Linear similarity correlation strong hai          | Cosine similarity reliable hai                     |
| Cosine Spearman           | 0.8203        | 0.75 – 0.90   | 🟢 Good          | Cosine similarity best performing metric hai      | Retrieval me cosine use karo (recommended)         |
| Manhattan Spearman        | 0.8194        | 0.70 – 0.85   | 🟢 Good          | Slightly lower but comparable                     | Rarely used in RAG                                 |
| Euclidean Spearman        | 0.8203        | 0.75 – 0.88   | 🟢 Good          | Cosine ke almost equal                            | Acceptable alternative distance metric             |


| Score Range | Interpretation          | Typical Model Type          |
| ----------- | ----------------------- | --------------------------- |
| < 0.70      | Weak semantic alignment | Poor / old embeddings       |
| 0.70 – 0.78 | Moderate                | Small generic models        |
| 0.78 – 0.83 | **Good Baseline**       | Lightweight models (MiniLM) |
| 0.83 – 0.88 | Strong                  | Fine-tuned mid-size models  |
| 0.88+       | Very Strong / Near SOTA | Large embedding models      |


In [ ]:
!pip install mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 34.3 MB/s eta 0:00:00


In [ ]:
import mteb
from typing import Dict, Any, Optional

def eval_stsbenchmark(
    model_name: str,
    batch_size: int = 32,
    normalize_embeddings: bool = True,
    languages: Optional[list[str]] = None,
) -> Dict[str, Any]:
    """
    Evaluate any embedding model on MTEB STSBenchmark and return key metrics.
    Works with model names supported by mteb.get_model().

    Returns a dict with main metrics + raw result object.
    """
    if languages is None:
        languages = ["eng"]

    # 1) Load model via MTEB
    model = mteb.get_model(model_name)

    # 2) Get task objects
    tasks = mteb.get_tasks(tasks=["STSBenchmark"], languages=languages)

    # 3) Run evaluation
    res = mteb.evaluate(
        model,
        tasks=tasks,
        encode_kwargs={
            "batch_size": batch_size,
            "normalize_embeddings": normalize_embeddings,
        },
    )

    # 4) Extract metrics cleanly
    # res.task_results -> list[TaskResult]
    tr = next((t for t in res.task_results if t.task_name == "STSBenchmark"), None)
    if tr is None:
        raise RuntimeError("STSBenchmark TaskResult not found in results.")

    # scores structure: {'test': [ { ...metrics... } ]}
    test_entry = tr.scores["test"][0]
    out = {
        "model_name": res.model_name,
        "model_revision": res.model_revision,
        "main_score": float(test_entry["main_score"]),
        "spearman": float(test_entry["spearman"]),
        "pearson": float(test_entry["pearson"]),
        "cosine_spearman": float(test_entry["cosine_spearman"]),
        "cosine_pearson": float(test_entry["cosine_pearson"]),
        "euclidean_spearman": float(test_entry["euclidean_spearman"]),
        "manhattan_spearman": float(test_entry["manhattan_spearman"]),
        "languages": test_entry.get("languages"),
        "hf_subset": test_entry.get("hf_subset"),
        "raw": res,  # full ModelResult (in case you need it)
    }

    # Pretty print (optional)
    print("\n=== STSBenchmark (MTEB) ===")
    print(f"Model: {out['model_name']}")
    print(f"Revision: {out['model_revision']}")
    print(f"Main (Spearman): {out['main_score']:.4f}")
    print(f"Spearman: {out['spearman']:.4f} | Pearson: {out['pearson']:.4f}")
    print(
        "Cosine Spearman: {cs:.4f} | Euclidean Spearman: {es:.4f} | Manhattan Spearman: {ms:.4f}".format(
            cs=out["cosine_spearman"],
            es=out["euclidean_spearman"],
            ms=out["manhattan_spearman"],
        )
    )

    return out

E5 = Embedding from E(verything) to E(verything)

Developed by:

Microsoft Research

Popular model name:

intfloat/e5-base-v2

BGE = BAAI General Embedding

Developed by:

Beijing Academy of Artificial Intelligence

Popular models:

BAAI/bge-base-en-v1.5

In [ ]:
r1 = eval_stsbenchmark("sentence-transformers/all-MiniLM-L6-v2")
r2 = eval_stsbenchmark("BAAI/bge-base-en-v1.5")
r3 = eval_stsbenchmark("intfloat/e5-base-v2")

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

train.jsonl.gz:   0%|          | 0.00/278k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

 {'query': 'query: ', 'document': 'passage: '}
/usr/local/lib/python3.12/dist-packages/mteb/models/sentence_transformer_wrapper.py:88: UserWarning: Model prompts specified, these will overwrite the default model prompts. Current prompts will be:
 {'query': 'query: ', 'document': 'passage: '}
  warnings.warn(msg)


Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]


=== STSBenchmark (MTEB) ===
Model: intfloat/e5-base-v2
Revision: 1c644c92ad3ba1efdad3f1451a637716616a20e8
Main (Spearman): 0.8548
Spearman: 0.8548 | Pearson: 0.8493
Cosine Spearman: 0.8548 | Euclidean Spearman: 0.8548 | Manhattan Spearman: 0.8549


In [ ]:
results = [r1, r2, r3]
for r in sorted(results, key=lambda x: x["main_score"], reverse=True):
    print(f"{r['model_name']}: {r['main_score']:.4f}")

BAAI/bge-base-en-v1.5: 0.8642
intfloat/e5-base-v2: 0.8548
sentence-transformers/all-MiniLM-L6-v2: 0.8203


## Training Code

In [5]:
from datasets import load_from_disk
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [6]:
# Load base model
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# with inbuilt dataset
# 2. Load dataset
# dataset = load_dataset("sentence-transformers/all-nli", "triplet")
# train_dataset = dataset["train"].select(range(100_000))
# eval_dataset = dataset["dev"]

In [7]:
import zipfile
import os

zip_path = "/content/pharma_ft_data_backup.zip"       # zip file ka path
extract_to = "unzipped_data"     # folder jaha extract karna hai

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Unzip complete")

Unzip complete


In [8]:
# Load YOUR custom dataset
train_dataset = load_from_disk("/content/unzipped_data/pharma_ft_data/train_pairs")

Arrow file = Apache Arrow columnar format

HuggingFace Datasets internally use karta hai:

Fast memory-mapped storage

Efficient column-based access

Faster training

Low RAM usage

Parallel data loading

Isliye:
Dataset ko JSON me store nahi karta
Arrow binary format me store karta hai.

In [9]:
print("Dataset loaded:", train_dataset)
print("Columns:", train_dataset.column_names)

Dataset loaded: Dataset({
    features: ['anchor', 'positive'],
    num_rows: 8
})
Columns: ['anchor', 'positive']


In [10]:
# Define loss (perfect for anchor-positive pairs)
loss = MultipleNegativesRankingLoss(model)

In [11]:
# Training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="models/pharma-embedding-ft",
    num_train_epochs=3,                  # increase since dataset small
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,                           # use False if CPU
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=10,
    save_strategy="epoch",
)

In [12]:
# Trainer (NO evaluator needed for pair dataset)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [13]:
# Train
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=0.15489602088928223, metrics={'train_runtime': 96.7209, 'train_samples_per_second': 0.248, 'train_steps_per_second': 0.031, 'total_flos': 0.0, 'train_loss': 0.15489602088928223, 'epoch': 3.0})

In [14]:
# Save model
model.save_pretrained("models/pharma-embedding-ft")

print("Fine-tuned model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully.


In [15]:
import os
import zipfile

def zip_folder_manual(folder_path: str, output_zip_path: str):
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)

    print(f"[OK] Folder zipped at: {output_zip_path}")


# Example
zip_folder_manual("/content/models", "model_backup.zip")

[OK] Folder zipped at: model_backup.zip


## Inference code

In [16]:
from sentence_transformers import SentenceTransformer

embeddings_model = SentenceTransformer("/content/models/pharma-embedding-ft")

sentences = [
    "What is Amoxycillin Capsules 500mg?"
]

embeddings = embeddings_model.encode(sentences, normalize_embeddings=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## now lets create a RAG and use the custom finetuned model

In [17]:
!pip install -U langchain
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.1/127.1 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.4
    Uninstalling langchain-1.3.4:
      Successfully uninstalled langchain-1.3.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 21.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.0
    Uninstalling langchain-core-1.4.0:
      Successfully uninstalled langchain-core-1.4.0


In [19]:
!pip install -qU langchain-community pypdf

In [20]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_2717/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [22]:
# Load document using PyPDFLoader document loader
loader = PyPDFLoader("/content/unzipped_data/pharma_demo.pdf")
documents = loader.load()

In [ ]:
documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-22T09:30:19+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-02-22T09:30:19+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': '/content/pharma_demo.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='PHARMA PRODUCT DOSSIER (SYNTHETIC)\nProduct: Amoxycillin Capsules 500 mg\nIndication: Treatment of bacterial infections (upper respiratory tract, urinary tract).\nMechanism of Action: Beta-lactam antibiotic; inhibits bacterial cell wall synthesis.\nContraindications: Hypersensitivity to penicillins; history of severe allergy.\nWarnings: Risk of anaphylaxis; monitor for rash; adjust dose in renal impairment.\nDosage & Administration:\n- Adults: 500 mg every 8 hours.\n- Renal impairment: dose adjustment required based on creatinine clearance.\nAdverse Reactions:\n- Common: nausea, diarrhea, rash.\n- Seriou

In [23]:
len(documents)

3

In [ ]:
# !pip install langchain-text-splitters

In [ ]:
# from langchain_text_splitters import CharacterTextSplitter
# splitter = CharacterTextSplitter(
#     separator="\n\n",
#     chunk_size=20,
#     chunk_overlap=15
# )

In [ ]:
# chunks = splitter.split_documents(documents)

In [ ]:
# chunks

In [ ]:
# print(len(chunks))

In [24]:
from langchain_community.vectorstores.faiss import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [25]:
# your fine-tuned SentenceTransformer model path
MODEL_PATH = "/content/models/pharma-embedding-ft"

# Embedding object (has embed_documents + embed_query)
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_PATH,
    model_kwargs={"device": "cuda"},          # change to "cpu" if needed
    encode_kwargs={"normalize_embeddings": True}
)

/tmp/ipykernel_2717/3771146459.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [36]:
repo_id = "shashrao54/embedding-finetuning-model"

In [39]:
import os
from google.colab import userdata
from huggingface_hub import login

# Retrieve token from Colab Secrets and log in
hf_token = userdata.get('HF_TOKEN_WRITE')
if hf_token:
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face!")
else:
    print("Error: HF_TOKEN secret not found in Colab.")

Successfully authenticated with Hugging Face!


In [42]:
embeddings_model.push_to_hub(
    repo_id,
    private=False
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._9x9ryb/model.safetensors:   4%|4         | 19.3MB /  438MB            

'https://huggingface.co/shashrao54/embedding-finetuning-model/commit/841f8237bc8c2cd6ac45c057f9c7a2b4635e96bf'

In [26]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 87.6 MB/s eta 0:00:00


In [27]:


# Build FAISS from your Document list
vectorstore = FAISS.from_documents(documents, embeddings)

# Test search
query = "What are the contraindications?"
results = vectorstore.similarity_search(query, k=3)

for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(r.page_content[:500])


--- Result 1 ---
PHARMA PRODUCT DOSSIER (SYNTHETIC)
Product: Amoxycillin Capsules 500 mg
Indication: Treatment of bacterial infections (upper respiratory tract, urinary tract).
Mechanism of Action: Beta-lactam antibiotic; inhibits bacterial cell wall synthesis.
Contraindications: Hypersensitivity to penicillins; history of severe allergy.
Warnings: Risk of anaphylaxis; monitor for rash; adjust dose in renal impairment.
Dosage & Administration:
- Adults: 500 mg every 8 hours.
- Renal impairment: dose adjustment r

--- Result 2 ---
MANUFACTURING & CMC (SYNTHETIC)
Manufacturing Process:
Blending -> Encapsulation -> In-process checks -> Packaging.
Critical process parameters: blend uniformity, capsule fill weight, moisture control.
Specifications:
- Identification: conforms to reference standard.
- Related substances: within limits.
- Microbial limits: complies with pharmacopeial requirements.
Packaging:
Alu-Alu blister packs; carton labeling includes batch number and expiry date.

--- Re

In [28]:
vectorstore.save_local("faiss_index_")

In [29]:
results = vectorstore.similarity_search("What are the warnings and precautions?", k=1)

In [30]:
for r in results:
    print("----")
    print(r.page_content)

----
PHARMA PRODUCT DOSSIER (SYNTHETIC)
Product: Amoxycillin Capsules 500 mg
Indication: Treatment of bacterial infections (upper respiratory tract, urinary tract).
Mechanism of Action: Beta-lactam antibiotic; inhibits bacterial cell wall synthesis.
Contraindications: Hypersensitivity to penicillins; history of severe allergy.
Warnings: Risk of anaphylaxis; monitor for rash; adjust dose in renal impairment.
Dosage & Administration:
- Adults: 500 mg every 8 hours.
- Renal impairment: dose adjustment required based on creatinine clearance.
Adverse Reactions:
- Common: nausea, diarrhea, rash.
- Serious: anaphylaxis, C. difficile-associated diarrhea (rare).


In [31]:
retriever = vectorstore.as_retriever()

In [ ]:
import os
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

In [32]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [33]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [34]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant.
Use the following context to answer the question. If you don't know, say you don't know.

Context:
{context}

Question:
{question}

Answer:"""
)

In [35]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

NameError: name 'llm' is not defined

In [ ]:
rag_chain.invoke("What are the contraindications for Amoxycillin 500 mg?")

In [ ]:
# What are the contraindications for Amoxycillin 500 mg?
# How does Amoxycillin work and how is it eliminated from the body?